In [1]:
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np

import os
import sys
import scipy.io
import click

import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import scipy.sparse as sp
import itertools
from tqdm import tqdm

os.chdir("/g/stegle/schrod/code/TCell")
from utils import *

logger = get_logger(__name__)


In [2]:
min_cells_per_bulk = 5
target_column = "target"
control_group = "NO-TARGET"
h5ad_file = f"/g/stegle/schrod/data/T_Cell/promoter_single_guide_hvg_raw_counts.h5ad"


In [3]:
logger.info(f"Read anndata object from: {h5ad_file}")
adata = ad.read_h5ad(os.path.join(h5ad_file))


[2026-02-16 15:25:37,450] INFO:utils: Read anndata object from: /g/stegle/schrod/data/T_Cell/promoter_single_guide_hvg_raw_counts.h5ad


In [4]:
logger.info(f"Build pseudobulk profiles by aggregating cells with the same guide per donor...")
obs_cols = ['donor', 'guide']   # Columns used to aggregate cells

# Build group keys and factorize
key_df = adata.obs[obs_cols].astype(str)

group_key = key_df.astype(str).agg('|'.join, axis=1)
codes, uniques = pd.factorize(group_key, sort=True)
n_obs = key_df.shape[0]
n_groups = len(uniques)

# weight pseudobulk contributions
weights = np.ones(n_obs, dtype=np.float64)

# One-hot membership matrix (n_obs x n_groups)
G = sp.csr_matrix(
    (weights, (np.arange(n_obs), codes)),
    shape=(n_obs, n_groups)
)
group_sizes = np.asarray(G.sum(axis=0)).ravel().astype(np.float64)

X_pb = G.T @ adata.X  # (groups x genes)

# Build obs for pseudobulks
obs_new = pd.DataFrame(index=uniques)
obs_new["donor_guide"] = obs_new.index
obs_new["donor"] = obs_new["donor_guide"].apply(lambda x: x.split('|')[0])
obs_new["guide"] = obs_new["donor_guide"].apply(lambda x: x.split('|')[1])
obs_new["target"] = obs_new["guide"].apply(lambda x: "-".join((x.split('-')[1:-1])))
split = obs_new.index.str.split('|', expand=True)
obs_new['n_cells'] = group_sizes.astype(float)

# Create pseudobulk AnnData
adata_pb = ad.AnnData(X_pb, obs=obs_new, var=adata.var.copy())
mapping_df = adata.obs[['target', 'target_idx']].drop_duplicates()
mapping = dict(zip(mapping_df['target'], mapping_df['target_idx']))
adata_pb.obs['target_idx'] = adata_pb.obs['target'].map(mapping)
adata_pb.var_names = adata.var_names.copy()
adata_pb = adata_pb[adata_pb.obs['n_cells']>=min_cells_per_bulk].copy()
logger.info(f"Discarded buks with less than {min_cells_per_bulk} cells. Remaining bulks: {adata_pb.shape[0]}")



[2026-02-16 15:29:45,023] INFO:utils: Build pseudobulk profiles by aggregating cells with the same guide per donor...
[2026-02-16 15:30:57,743] INFO:utils: Discarded buks with less than 5 cells. Remaining bulks: 48700


In [5]:
logger.info(f"Save norm log pseudobulk data...")
norm_log(adata_pb)
adata_pb.write_h5ad(f"/g/stegle/schrod/data/T_Cell/promoter_bulk_profiles_1guide.h5ad")



[2026-02-16 15:30:57,790] INFO:utils: Save norm log pseudobulk data...


In [9]:
adata_pb_nt = adata_pb[adata_pb.obs.target=="NO-TARGET"]

In [11]:
adata_pb_nt.write_h5ad(f"/g/stegle/schrod/data/T_Cell/promoter_bulk_profiles_1guide_non_targeting.h5ad")
